# ILDC × PredEx Overlap Removal

Removes PredEx test cases that also appear in the ILDC train/dev splits.

**Why:** PredEx and ILDC draw from the same Indian Supreme Court corpus. Cases shared between ILDC train/dev and PredEx test = data leakage.

**Matching strategy:** ILDC text has OCR artifacts (`court` → `companyrt`, `company` → `companypany`). PredEx has clean text. Direct string match fails. Fix: strip all non-alphanumeric characters from both sides, then match on sliding 300-char windows. A single window hit = same case.

## 0. Config — set your file paths here

In [ ]:
ILDC_TRAIN_PATH = "criminal_train.parquet"
ILDC_DEV_PATH   = "criminal_dev.parquet"
PREDEX_PATH     = "predex_test_final.csv"
OUTPUT_PATH     = "predex_test_deduped.csv"
REPORT_PATH     = "overlap_report.csv"

# Fingerprint window settings
WINDOW_SIZE    = 300   # chars per window
WINDOW_STEP    = 150   # step between windows
MIN_TEXT_START = 100   # skip first N chars (headers differ between datasets)

## 1. Imports

In [ ]:
import re
import pandas as pd

## 2. Load files

In [ ]:
ildc_train = pd.read_parquet(ILDC_TRAIN_PATH)
ildc_dev   = pd.read_parquet(ILDC_DEV_PATH)
ildc_all   = pd.concat([ildc_train, ildc_dev], ignore_index=True)

print(f"ILDC train : {len(ildc_train):,} cases")
print(f"ILDC dev   : {len(ildc_dev):,} cases")
print(f"ILDC total : {len(ildc_all):,} cases")
ildc_all.head(2)

In [ ]:
predex = pd.read_csv(PREDEX_PATH)
print(f"PredEx rows: {len(predex):,}")
predex.head(2)

## 3. Fingerprint helpers

In [ ]:
def clean(text: str) -> str:
    """Strip everything except lowercase alphanumeric — neutralises OCR noise."""
    return re.sub(r'[^a-z0-9]', '', str(text).lower())

def get_windows(text: str) -> list:
    """Return all fixed-size sliding windows from the cleaned text."""
    c = clean(text)
    windows = []
    for start in range(MIN_TEXT_START, len(c) - WINDOW_SIZE, WINDOW_STEP):
        windows.append(c[start : start + WINDOW_SIZE])
    return windows

## 4. Build ILDC fingerprint index

Maps every 300-char window → `ildc_id`. Takes ~1–2 min for the full train set.

In [ ]:
print("Building index (this takes a minute) ...")

ildc_index = {}
for _, row in ildc_all.iterrows():
    for w in get_windows(row["text"]):
        ildc_index[w] = row["id"]

print(f"Index size: {len(ildc_index):,} windows")

## 5. Find overlapping PredEx cases

In [ ]:
def find_match(predex_input: str) -> str:
    """Return matching ILDC id if found, else None."""
    for w in get_windows(predex_input):
        if w in ildc_index:
            return ildc_index[w]
    return None

predex["_matched_ildc_id"] = predex["Input"].apply(lambda x: find_match(str(x)))
overlap_mask = predex["_matched_ildc_id"].notna()

print(f"Overlapping cases : {overlap_mask.sum()}")
print(f"Clean cases       : {(~overlap_mask).sum()}")

## 6. Inspect overlaps

In [ ]:
overlap_df = predex[overlap_mask][["Case Name", "_matched_ildc_id"]].copy()
overlap_df.columns = ["predex_case_name", "ildc_id"]
overlap_df

## 7. Save outputs

In [ ]:
# Save overlap report
overlap_df.to_csv(REPORT_PATH, index=False)
print(f"Overlap report saved to: {REPORT_PATH}")

# Save clean PredEx
predex_clean = predex[~overlap_mask].drop(columns=["_matched_ildc_id"]).reset_index(drop=True)
predex_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Clean PredEx saved to  : {OUTPUT_PATH}")
print(f"Final row count        : {len(predex_clean):,}")